# Example 3.6 / 1: A production LP

Companion notebook for **3.6 Duals and Reduced Costs** in the AMPL lab Documentation, section "A production LP". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## A production LP

Assume `profit` is dollars per ton, `Make` is tons, and `avail` is hours. The objective is
dollars and the `Time` dual is therefore dollars per additional hour under the current LP
basis and sign convention.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;
param rate {PROD} > 0;
param profit {PROD};
param market {PROD} >= 0;
param avail >= 0;
var Make {p in PROD} >= 0, <= market[p];
maximize Total_Profit: sum {p in PROD} profit[p] * Make[p];
subject to Time: sum {p in PROD} Make[p] / rate[p] <= avail;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils"]
ampl.param["rate"] = {"bands": 200, "coils": 140}
ampl.param["profit"] = {"bands": 25, "coils": 30}
ampl.param["market"] = {"bands": 6000, "coils": 4000}
ampl.param["avail"] = 40

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.solve_result)
print("Time dual:", ampl.con["Time"].dual())
print("Make values:", ampl.var["Make"].to_dict())
print("Reduced costs:", ampl.var["Make"].get_values(["rc"]).to_dict())

HiGHS 1.14.0: optimal solution; objective 192000
0 simplex iterations
0 barrier iterations
solved
Time dual: 4200.0
Make values: {'bands': 6000, 'coils': 1400}
Reduced costs: {'bands': 4, 'coils': 0}


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PROD := bands coils;
param avail := 40;
param: rate profit market :=
  bands 200 25 6000
  coils 140 30 4000;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display solve_result, Time.dual;
display Make, Make.rc;

HiGHS 1.14.0: optimal solution; objective 192000
0 simplex iterations
0 barrier iterations
solve_result = solved
Time.dual = 4200

:       Make Make.rc    :=
bands   6000     4
coils   1400     0
;

